# 1) Load Raw Dataset + Initial Quality Check

Goal:
- Load the raw scraped CSV (do not modify raw file)
- Confirm dataset size and columns
- Measure missingness (what fields are incomplete)
- Check duplicates (listing_id / url)
- Quick numeric sanity check (price, area, beds, baths)

This tells us whether the scrape is usable before we clean.

In [3]:
import pandas as pd
import numpy as np
import os

RAW_PATH = "../data/raw/dubizzle_apartments_raw.csv" 
df = pd.read_csv(RAW_PATH)

print("Raw shape:", df.shape)
display(df.head(3))

print("\nColumns:")
print(list(df.columns))

# Missingness %
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("\nMissingness % (top 20):")
display(missing_pct.head(20))

# Duplicate counts (safe if columns exist)
dup_id = df.duplicated(subset=["listing_id"]).sum() if "listing_id" in df.columns else None
dup_url = df.duplicated(subset=["url"]).sum() if "url" in df.columns else None
print(f"\nDuplicates -> listing_id: {dup_id}, url: {dup_url}")

# Coerce numeric columns
num_cols = ["price_egp", "area_sqm", "bedrooms", "bathrooms", "amenities_count"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print("\nNumeric summary:")
display(df[[c for c in num_cols if c in df.columns]].describe(percentiles=[.01, .05, .5, .95, .99]))

Raw shape: (4974, 14)


,listing_id,url,price_egp,location_text,type,ownership,area_sqm,bedrooms,bathrooms,furnished,payment_option,completion_status,amenities,amenities_count
0,503203361,https://www.dubizzle.com.eg/en/ad/apartment-fo...,3100000.0,"Sheikh Zayed, Giza",Apartment,Resale,71.0,2.0,1.0,No,Cash,Ready,Water Meter|Electricity Meter,2
1,503171146,https://www.dubizzle.com.eg/en/ad/apartment-3-...,1180000.0,"Nakheel, Alexandria",Apartment,Primary,140.0,3.0,2.0,No,Cash or Installment,Ready,Electricity Meter|Water Meter|Covered Parking|...,5
2,502962823,https://www.dubizzle.com.eg/en/ad/apartment-fo...,11250000.0,"Smoha, Alexandria",Apartment,Resale,244.0,3.0,2.0,No,Cash,Ready,Security|Electricity Meter|Water Meter|Balcony...,6



Columns:
['listing_id', 'url', 'price_egp', 'location_text', 'type', 'ownership', 'area_sqm', 'bedrooms', 'bathrooms', 'furnished', 'payment_option', 'completion_status', 'amenities', 'amenities_count']

Missingness % (top 20):


amenities            23.763571
furnished            10.172899
ownership             0.361882
payment_option        0.341777
bathrooms             0.120627
type                  0.100523
area_sqm              0.100523
bedrooms              0.100523
completion_status     0.100523
price_egp             0.080418
location_text         0.080418
listing_id            0.000000
url                   0.000000
amenities_count       0.000000
dtype: float64


Duplicates -> listing_id: 0, url: 0

Numeric summary:


,price_egp,area_sqm,bedrooms,bathrooms,amenities_count
count,4.970000e+03,4969.000000,4969.000000,4968.000000,4974.000000
mean,1.116066e+07,166.634534,2.776011,2.258454,3.787495
std,1.574694e+08,81.236752,0.748549,0.877216,2.443588
min,3.500000e+05,26.000000,1.000000,1.000000,0.000000
1%,8.400000e+05,60.000000,1.000000,1.000000,0.000000
5%,1.472500e+06,85.000000,2.000000,1.000000,0.000000
50%,6.700000e+06,155.000000,3.000000,2.000000,5.000000
95%,1.700000e+07,280.000000,4.000000,4.000000,6.000000
99%,2.715500e+07,441.600000,5.000000,4.000000,6.000000
max,8.000000e+09,3000.000000,10.000000,10.000000,6.000000


# 2) Core Cleaning (Dedup + Required Fields + Sanity Filters)

We create a working copy (`df0`) and apply **defensible** cleaning rules:

1) Deduplicate:
- keep first row per `listing_id`
- keep first row per `url`

2) Require critical features needed for modeling:
- `price_egp`, `area_sqm`, `location_text`

3) Sanity filters (wide ranges to avoid removing real data):
- price_egp: 50,000 → 200,000,000
- area_sqm: 15 → 2000
- bedrooms/bathrooms: if present, keep 0 → 15

We print how many rows remain after cleaning.

In [4]:
df0 = df.copy()

# Dedup
if "listing_id" in df0.columns:
    df0 = df0.drop_duplicates(subset=["listing_id"], keep="first")
if "url" in df0.columns:
    df0 = df0.drop_duplicates(subset=["url"], keep="first")

# Require essentials for a price model
required = [c for c in ["price_egp", "area_sqm", "location_text"] if c in df0.columns]
df0 = df0.dropna(subset=required)

# Sanity filters (wide)
df0 = df0[(df0["price_egp"] >= 50_000) & (df0["price_egp"] <= 200_000_000)]
df0 = df0[(df0["area_sqm"] >= 15) & (df0["area_sqm"] <= 2000)]

if "bedrooms" in df0.columns:
    df0 = df0[(df0["bedrooms"].isna()) | ((df0["bedrooms"] >= 0) & (df0["bedrooms"] <= 15))]
if "bathrooms" in df0.columns:
    df0 = df0[(df0["bathrooms"].isna()) | ((df0["bathrooms"] >= 0) & (df0["bathrooms"] <= 15))]

print("After core cleaning:", df0.shape)
display(df0.head(3))

After core cleaning: (4965, 14)


,listing_id,url,price_egp,location_text,type,ownership,area_sqm,bedrooms,bathrooms,furnished,payment_option,completion_status,amenities,amenities_count
0,503203361,https://www.dubizzle.com.eg/en/ad/apartment-fo...,3100000.0,"Sheikh Zayed, Giza",Apartment,Resale,71.0,2.0,1.0,No,Cash,Ready,Water Meter|Electricity Meter,2
1,503171146,https://www.dubizzle.com.eg/en/ad/apartment-3-...,1180000.0,"Nakheel, Alexandria",Apartment,Primary,140.0,3.0,2.0,No,Cash or Installment,Ready,Electricity Meter|Water Meter|Covered Parking|...,5
2,502962823,https://www.dubizzle.com.eg/en/ad/apartment-fo...,11250000.0,"Smoha, Alexandria",Apartment,Resale,244.0,3.0,2.0,No,Cash,Ready,Security|Electricity Meter|Water Meter|Balcony...,6


# 3) Feature Preparation (Location Parsing + Light Normalization)

We convert `location_text` into more ML-friendly fields:

- `district`: typically the last segment after a comma
- `city`: only if the last segment is one of {Cairo, Giza, Alexandria}

Examples:
- "Heliopolis, Cairo" → district=Heliopolis, city=Cairo
- "Lake View Residence Compound, 5th Settlement" → district=5th Settlement, city=NaN

We also:
- trim categorical strings
- standardize `furnished` (fill missing with "Unknown")

Finally, we review the most common districts (helps understand dataset coverage).

In [5]:
KNOWN_CITIES = {"Cairo", "Giza", "Alexandria"}

def parse_location(loc: str):
    if pd.isna(loc):
        return pd.Series([np.nan, np.nan])
    parts = [p.strip() for p in str(loc).split(",") if p.strip()]
    if not parts:
        return pd.Series([np.nan, np.nan])

    # If last token is a known city -> city=last, district=previous
    if parts[-1] in KNOWN_CITIES:
        city = parts[-1]
        district = parts[-2] if len(parts) >= 2 else np.nan
    else:
        city = np.nan
        district = parts[-1]
    return pd.Series([city, district])

df0[["city", "district"]] = df0["location_text"].apply(parse_location)

# Light normalization for categoricals
cat_cols = ["type", "ownership", "furnished", "payment_option", "completion_status", "city", "district"]
for c in cat_cols:
    if c in df0.columns:
        df0[c] = df0[c].astype("string").str.strip()

if "furnished" in df0.columns:
    df0["furnished"] = df0["furnished"].fillna("Unknown")

print("Location parsing sample:")
display(df0[["location_text", "district", "city"]].head(12))

print("\nTop districts:")
display(df0["district"].value_counts().head(15))

Location parsing sample:


,location_text,district,city
0,"Sheikh Zayed, Giza",Sheikh Zayed,Giza
1,"Nakheel, Alexandria",Nakheel,Alexandria
2,"Smoha, Alexandria",Smoha,Alexandria
3,"Smoha, Alexandria",Smoha,Alexandria
4,"Mountain View iCity Compound, 6th of October",6th of October,<NA>
5,"EL Patio ORO Compound, 5th Settlement",5th Settlement,<NA>
6,"Fifth Square Compound, 5th Settlement",5th Settlement,<NA>
7,"Kafr Abdo, Alexandria",Kafr Abdo,Alexandria
8,"Taj City Compound, 1st Settlement",1st Settlement,<NA>
9,"Nakheel, Alexandria",Nakheel,Alexandria



Top districts:


district
5th Settlement     735
Sheikh Zayed       671
6th of October     668
Smoha              389
Nakheel            219
Madinaty           168
Hadayek October    139
Mostakbal City     136
1st Settlement     130
Moharam Bik        104
New Cairo          101
Miami               80
Laurent             79
Sidi Beshr          76
Agami               72
Name: count, dtype: Int64

# 4) Outlier Handling + Save Processed Dataset

We compute a useful derived feature:

- `price_per_sqm = price_egp / area_sqm`

Then we remove extreme tails to reduce noise:
- drop bottom 1% and top 1% of `price_per_sqm`

This helps stabilize training metrics and prevents extreme outliers from dominating the model.

Finally we save the cleaned dataset to:

`data/processed/dubizzle_apartments_clean.csv`

This is the dataset we will use for model training.

In [7]:
df0["price_per_sqm"] = df0["price_egp"] / df0["area_sqm"]

low, high = df0["price_per_sqm"].quantile([0.01, 0.99])
df1 = df0[(df0["price_per_sqm"] >= low) & (df0["price_per_sqm"] <= high)].copy()

print("Before outlier trim:", df0.shape)
print("After outlier trim :", df1.shape)
print("price_per_sqm 1%..99% bounds:", (low, high))

os.makedirs("data/processed", exist_ok=True)
OUT_PATH = "../data/processed/dubizzle_apartments_clean.csv"
df1.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

display(df1[["price_egp","area_sqm","bedrooms","bathrooms","district","city","price_per_sqm"]].head(10))

Before outlier trim: (4965, 17)
After outlier trim : (4871, 17)
price_per_sqm 1%..99% bounds: (7416.666666666667, 138341.11787493076)
Saved: ../data/processed/dubizzle_apartments_clean.csv


,price_egp,area_sqm,bedrooms,bathrooms,district,city,price_per_sqm
0,3100000.0,71.0,2.0,1.0,Sheikh Zayed,Giza,43661.971831
1,1180000.0,140.0,3.0,2.0,Nakheel,Alexandria,8428.571429
2,11250000.0,244.0,3.0,2.0,Smoha,Alexandria,46106.557377
3,17400000.0,290.0,4.0,3.0,Smoha,Alexandria,60000.000000
4,6900000.0,140.0,3.0,3.0,6th of October,<NA>,49285.714286
5,13500000.0,210.0,3.0,2.0,5th Settlement,<NA>,64285.714286
6,14000000.0,115.0,1.0,1.0,5th Settlement,<NA>,121739.130435
7,6000000.0,145.0,3.0,2.0,Kafr Abdo,Alexandria,41379.310345
8,10000000.0,207.0,4.0,4.0,1st Settlement,<NA>,48309.178744
9,890000.0,100.0,2.0,1.0,Nakheel,Alexandria,8900.000000


# Step 5 — Final Price Prediction Model (Monotonic Gradient Boosting)

This step trains the final machine learning model used for predicting real estate prices.

The model is based on **HistGradientBoostingRegressor**, a tree-based ensemble algorithm that performs well on structured/tabular data and can capture nonlinear relationships between features and property prices.

Several improvements are incorporated to produce realistic and economically consistent predictions:

### Feature Engineering
The following engineered features are used:

- **area_squared** — captures nonlinear effects of property size
- **rooms_total** — combined bedroom and bathroom count
- **rooms_per_100sqm** — layout density indicator
- **amenities_density** — amenities relative to property size
- **is_compound** — detects whether the property is located in a compound

### District Grouping
To reduce sparsity in categorical features, only the **top 25 most frequent districts** are preserved individually while the rest are grouped into **"Other"**.

### Monotonic Constraints
To ensure economically plausible predictions, **monotonic constraints** are applied to key structural variables:

- Area
- Bedrooms
- Bathrooms
- Amenities
- Compound presence

This guarantees that increasing these attributes **cannot decrease the predicted property price**, preventing unrealistic model behavior.

### Target Transformation
The model is trained using **log(price)** to reduce skew in the target variable. Predictions are later converted back to Egyptian Pounds.

### Evaluation Metrics
Model performance is evaluated using:

- **MAE (Mean Absolute Error)**
- **RMSE (Root Mean Squared Error)**
- **R² (Coefficient of Determination)**

All metrics are reported on the **original price scale (EGP)**.

In [27]:
# ============================================================
# STEP 5 — Final Model (Monotonic Constraints + Structural Features)
# Predict TOTAL PRICE (EGP) with log-target training
# Ensures: increasing bathrooms/bedrooms/area/amenities/compound cannot DECREASE price
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

# -----------------------------
# 0) Prepare modeling dataframe
# -----------------------------
df_model = df1.copy()

categorical_cols = ["district", "type", "ownership", "furnished", "payment_option", "completion_status"]
numeric_base = ["area_sqm", "bedrooms", "bathrooms"]

# Ensure categorical columns exist + normalize
for col in categorical_cols:
    if col not in df_model.columns:
        df_model[col] = "Unknown"
    df_model[col] = df_model[col].fillna("Unknown").astype("string").str.strip()

# Coerce numeric columns
for col in numeric_base:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

# Ensure amenities_count exists
if "amenities_count" not in df_model.columns:
    df_model["amenities_count"] = 0
df_model["amenities_count"] = pd.to_numeric(df_model["amenities_count"], errors="coerce").fillna(0)

# Ensure location_text exists (for compound flag)
if "location_text" not in df_model.columns:
    df_model["location_text"] = ""
df_model["location_text"] = df_model["location_text"].fillna("").astype("string")

# Drop essentials
df_model = df_model.dropna(subset=["price_egp", "area_sqm"])

# -----------------------------
# 1) District grouping (rare -> Other)
# -----------------------------
TOP_DISTRICTS = 25
top_districts = df_model["district"].value_counts().head(TOP_DISTRICTS).index
df_model["district_grouped"] = df_model["district"].where(df_model["district"].isin(top_districts), "Other")

categorical_cols_improved = ["district_grouped", "type", "ownership", "furnished", "payment_option", "completion_status"]

# -----------------------------
# 2) Feature engineering
# -----------------------------
df_model["bedrooms_f"] = df_model["bedrooms"].fillna(0)
df_model["bathrooms_f"] = df_model["bathrooms"].fillna(0)

df_model["area_squared"] = df_model["area_sqm"] ** 2
df_model["rooms_total"] = df_model["bedrooms_f"] + df_model["bathrooms_f"]
df_model["rooms_per_100sqm"] = (df_model["rooms_total"] / df_model["area_sqm"]) * 100

df_model["amenities_density"] = df_model["amenities_count"] / df_model["area_sqm"]
df_model["is_compound"] = df_model["location_text"].str.contains("compound", case=False, na=False).astype(int)

# IMPORTANT: this order matters for constraints
numeric_cols_improved = [
    "area_sqm",
    "area_squared",
    "bedrooms_f",
    "bathrooms_f",
    "rooms_total",
    "rooms_per_100sqm",
    "amenities_count",
    "amenities_density",
    "is_compound"
]

X = df_model[numeric_cols_improved + categorical_cols_improved]
y = df_model["price_egp"].astype(float)

# -----------------------------
# 3) Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

y_train_log = np.log1p(y_train)

# -----------------------------
# 4) Preprocessor
# -----------------------------
ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, categorical_cols_improved),
        ("num", "passthrough", numeric_cols_improved),
    ],
    sparse_threshold=0
)

# ---- Fit preprocessor once to determine final feature count (needed for monotonic constraints)
preprocessor.fit(X_train)

# Number of features after one-hot
n_cat = len(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols_improved))
n_num = len(numeric_cols_improved)
total_features = n_cat + n_num

# -----------------------------
# 5) Monotonic constraints (must match transformed feature order)
# -----------------------------
# ColumnTransformer outputs: [all one-hot categorical features..., then numeric passthrough...]
cat_cst = [0] * n_cat  # no monotonic constraint for one-hot columns

# Constraints for numeric features in EXACT numeric_cols_improved order:
num_cst = [
    1,  # area_sqm
    1,  # area_squared
    1,  # bedrooms_f
    1,  # bathrooms_f
    1,  # rooms_total
    0,  # rooms_per_100sqm (leave unconstrained)
    1,  # amenities_count
    1,  # amenities_density
    1   # is_compound
]

monotonic_constraints = cat_cst + num_cst

print("Transformed feature count:", total_features)
print("Monotonic constraints len:", len(monotonic_constraints))

# -----------------------------
# 6) Model (Monotonic HistGradientBoosting)
# -----------------------------
model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=9,
    max_iter=500,
    min_samples_leaf=20,
    random_state=42,
    monotonic_cst=monotonic_constraints
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train_log)

# Predict log(price) -> EGP
pred_log = pipeline.predict(X_test)
pred_egp = np.expm1(pred_log)

# -----------------------------
# 7) Metrics (EGP scale)
# -----------------------------
mae = mean_absolute_error(y_test, pred_egp)
rmse = np.sqrt(mean_squared_error(y_test, pred_egp))
r2 = r2_score(y_test, pred_egp)

print("\nFinal Model (Monotonic Constraints) — EGP evaluation")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

print("\nDistrict grouping:")
print("Unique districts original:", df_model["district"].nunique())
print("Unique districts grouped :", df_model["district_grouped"].nunique())
print("Top districts kept:", TOP_DISTRICTS)

# Quick sample
sample = pd.DataFrame({
    "actual_egp": y_test.values[:10],
    "pred_egp": pred_egp[:10]
})
sample["abs_error"] = (sample["actual_egp"] - sample["pred_egp"]).abs()
display(sample)

import joblib
from pathlib import Path

# Go to project root, then create /models
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

joblib.dump(
    {
        "pipeline": pipeline,
        "categorical_cols": categorical_cols_improved,
        "numeric_cols": numeric_cols_improved,
        "top_districts": list(top_districts),
        "top_districts_n": TOP_DISTRICTS
    },
    models_dir / "dubizzle_price_model.joblib"
)

print("Saved:", models_dir / "dubizzle_price_model.joblib")

Transformed feature count: 54
Monotonic constraints len: 54

Final Model (Monotonic Constraints) — EGP evaluation
MAE : 2016207.8614565553
RMSE: 3084325.884335314
R²  : 0.5517902322560979

District grouping:
Unique districts original: 89
Unique districts grouped : 26
Top districts kept: 25


,actual_egp,pred_egp,abs_error
0,11100000.0,9.955734e+06,1.144266e+06
1,10500000.0,6.782394e+06,3.717606e+06
2,2000000.0,2.027137e+06,2.713659e+04
3,1180000.0,9.025795e+05,2.774205e+05
4,3700000.0,4.518076e+06,8.180760e+05
5,890000.0,9.025795e+05,1.257947e+04
6,11250000.0,8.248011e+06,3.001989e+06
7,14500000.0,7.831315e+06,6.668685e+06
8,2200000.0,1.866786e+06,3.332142e+05
9,6000000.0,5.865548e+06,1.344524e+05


Saved: ../models/dubizzle_price_model.joblib


In [24]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

# Custom scoring function that converts log predictions -> EGP before scoring
def r2_on_egp(estimator, X, y_true_egp):
    pred_log = estimator.predict(X)
    pred_egp = np.expm1(pred_log)
    return r2_score(y_true_egp, pred_egp)

perm = permutation_importance(
    pipeline,
    X_test,
    y_test,               # y_test is EGP
    n_repeats=10,
    random_state=42,
    scoring=r2_on_egp,     # IMPORTANT: correct scale
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

display(importance_df)

,feature,importance_mean,importance_std
0,area_sqm,0.260840,0.021216
9,district_grouped,0.250071,0.043499
3,bathrooms_f,0.226174,0.020238
8,is_compound,0.188659,0.030485
5,rooms_per_100sqm,0.096221,0.014452
13,payment_option,0.058237,0.011989
11,ownership,0.039990,0.007210
14,completion_status,0.019472,0.004255
12,furnished,0.016647,0.003895
10,type,0.015318,0.005851


In [21]:
# --------------------------------------
# Sensitivity Test: Bedrooms vs Bathrooms
# --------------------------------------

# Take a real sample from test set
sample = X_test.iloc[0].copy()

print("Original sample values:")
display(sample)

def predict_price(input_row):
    pred_log = pipeline.predict(pd.DataFrame([input_row]))
    return float(np.expm1(pred_log)[0])

base_price = predict_price(sample)

print(f"\nBase prediction: {base_price:,.0f} EGP")

# Increase bedrooms by 1
sample_bed = sample.copy()
sample_bed["bedrooms_f"] += 1
sample_bed["rooms_total"] = sample_bed["bedrooms_f"] + sample_bed["bathrooms_f"]
sample_bed["rooms_per_100sqm"] = (sample_bed["rooms_total"] / sample_bed["area_sqm"]) * 100

price_bed = predict_price(sample_bed)

# Increase bathrooms by 1
sample_bath = sample.copy()
sample_bath["bathrooms_f"] += 1
sample_bath["rooms_total"] = sample_bath["bedrooms_f"] + sample_bath["bathrooms_f"]
sample_bath["rooms_per_100sqm"] = (sample_bath["rooms_total"] / sample_bath["area_sqm"]) * 100

price_bath = predict_price(sample_bath)

print("\nAfter +1 Bedroom:")
print(f"Prediction: {price_bed:,.0f} EGP")
print(f"Change     : {price_bed - base_price:,.0f} EGP")

print("\nAfter +1 Bathroom:")
print(f"Prediction: {price_bath:,.0f} EGP")
print(f"Change     : {price_bath - base_price:,.0f} EGP")

Original sample values:


area_sqm                           198.0
area_squared                     39204.0
bedrooms_f                           3.0
bathrooms_f                          3.0
rooms_total                          6.0
rooms_per_100sqm                3.030303
amenities_count                        6
amenities_density               0.030303
is_compound                            0
district_grouped             Moharam Bik
type                           Apartment
ownership                        Primary
furnished                             No
payment_option       Cash or Installment
completion_status                  Ready
Name: 1360, dtype: object


Base prediction: 9,955,734 EGP

After +1 Bedroom:
Prediction: 10,587,552 EGP
Change     : 631,818 EGP

After +1 Bathroom:
Prediction: 10,706,597 EGP
Change     : 750,863 EGP


# Step 6 — Cross-Validation Model Evaluation

To verify that the model generalizes well across the dataset, **5-fold cross-validation** is performed.

Instead of relying on a single train/test split, cross-validation repeatedly trains the model on different subsets of the data and evaluates it on unseen folds.

This provides a more reliable estimate of model performance.

For each fold we compute:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² score

The final results are reported as the **mean and standard deviation across folds**, indicating the stability of the model.

This evaluation ensures that the model's predictive performance is not dependent on a single random split of the dataset.

In [23]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor


# Helper: build a fresh pipeline for each fold (so monotonic_cst matches feature count)
def build_monotonic_pipeline(X_train_fold, categorical_cols_improved, numeric_cols_improved):
    # 1) Fit a temporary preprocessor to learn how many one-hot columns this fold produces
    tmp_ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False)
    tmp_pre = ColumnTransformer(
        transformers=[
            ("cat", tmp_ohe, categorical_cols_improved),
            ("num", "passthrough", numeric_cols_improved),
        ],
        sparse_threshold=0
    )
    tmp_pre.fit(X_train_fold)

    n_cat = len(tmp_pre.named_transformers_["cat"].get_feature_names_out(categorical_cols_improved))

    # 2) Build monotonic constraints: 0 for all one-hot, then numeric constraints
    cat_cst = [0] * n_cat
    num_cst = [
        1,  # area_sqm
        1,  # area_squared
        1,  # bedrooms_f
        1,  # bathrooms_f
        1,  # rooms_total
        0,  # rooms_per_100sqm
        1,  # amenities_count
        1,  # amenities_density
        1   # is_compound
    ]
    monotonic_constraints = cat_cst + num_cst

    # 3) Create a fresh preprocessor (same config) + model with fold-correct monotonic_cst
    ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False)
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", ohe, categorical_cols_improved),
            ("num", "passthrough", numeric_cols_improved),
        ],
        sparse_threshold=0
    )

    model = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=9,
        max_iter=500,
        min_samples_leaf=20,
        random_state=42,
        monotonic_cst=monotonic_constraints
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])


# -----------------------------
# 5-Fold Cross Validation
# -----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores, rmse_scores, r2_scores = [], [], []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
    y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

    y_train_log = np.log1p(y_train_cv)

    model_cv = build_monotonic_pipeline(X_train_cv, categorical_cols_improved, numeric_cols_improved)
    model_cv.fit(X_train_cv, y_train_log)

    pred_log = model_cv.predict(X_test_cv)
    pred_egp = np.expm1(pred_log)

    mae_scores.append(mean_absolute_error(y_test_cv, pred_egp))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test_cv, pred_egp)))
    r2_scores.append(r2_score(y_test_cv, pred_egp))

print("Cross-Validation Results (5-Fold)\n")
print(f"MAE  : {np.mean(mae_scores):,.0f} ± {np.std(mae_scores):,.0f}")
print(f"RMSE : {np.mean(rmse_scores):,.0f} ± {np.std(rmse_scores):,.0f}")
print(f"R²   : {np.mean(r2_scores):.3f} ± {np.std(r2_scores):.3f}")

Cross-Validation Results (5-Fold)

MAE  : 2,010,906 ± 58,392
RMSE : 3,208,396 ± 201,250
R²   : 0.577 ± 0.022


Using 5-fold cross-validation, the final model achieved an average R² of 0.577 (± 0.022), indicating that it explains around 58% of the variance in property prices across different splits of the dataset. The average prediction error was MAE = 2.01M EGP (± 0.06M) and RMSE = 3.21M EGP (± 0.20M). The relatively small standard deviations suggest the model generalizes consistently and is not dependent on a single train/test split.